In [0]:


import sys
import io
import os
import pandas as pd
import numpy as np
import logging
from scipy.stats import chi2_contingency, spearmanr
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN
# ══════════════════════════════════════════════════════════════════════════════

INPUT_FILE   = 'KMC-400-20y-libro  45-DatosCompletos.xlsx'
OUT_INTERIM  = 'kmc_interim.csv'             # input para v9a_eda.py
OUT_MASTER   = 'kmc_master_completo.csv'
OUT_MODELO   = 'kmc_modelado_completo.csv'
OUT_TIPOS    = 'kmc_diccionario_tipos.csv'
OUT_MISSING  = 'kmc_diccionario_missing.csv'
OUT_REPORTE  = 'missing_reporte.csv'
OUT_REVISION = 'missing_para_revision.csv'
LOG_FILE     = 'pipeline_v9b.log'

WASI_CLINICAL_RANGE = (40, 160)
IQR_MULT            = 3.0
ICV_COLUMN          = 'ASEG_EstimatedTotalIntraCranialV'
MRI_QUALITY_FLAGS   = ['NEURO_tieneT1estructuralOK', 'NEURO_tieneDTIOK']
CORR_THRESHOLD      = 0.90
MISSING_DIFF_P      = 0.05
MAX_UNIQUE_CAT      = 10
MIN_DATOS_CORR      = 10
OUTCOME_THRESH      = 0.15

FORZAR_CONTINUA = [
    'TAP_AL_EXT0_CLIN', 'TAP_GO1_ERT0_CLIN', 'TAP_IC_EXT0_CLIN',
    'TAP_IC_EXT1_CLIN', 'TAP_IC_EXT2_CLIN', 'TAP_WM2_ERT0_CLIN',
    'TAP_WM2_OMT0_CLIN', 'TAP_WM2_MDT0_CLIN', 'TAP_WM2_STT0_CLIN',
    'TAP_DA3_OMT0_CLIN', 'TAP_DA3_OMT1_CLIN', 'TAP_DA3_OMT2_CLIN',
    'TAP_SA2_ERT3_CLIN', 'TAP_SA2_OMT3_CLIN', 'TAP_GO1_MDT0_CLIN',
    'TAP_GO1_STT0_CLIN', 'TAP_CS_EXT0_CLIN', 'TAP_CS_EXT1_CLIN',
    'TAP_CS_EXT2_CLIN',
]

VARS_TEXTO_LIBRE = [
    'NHPT_observacionesNHPT',
    'WASI_Observaciones',
    'CVLT_Observaciones',
    'PEDIATRIC_Observaciones3',
    'PEDIATRIC_Observaciones',
    'NEURO_COMENTARIO',
    'NUT_Fecha',
    'BirthDate',
    'Group',
    'SURV24_serpositivoaquienacudeparatomarladecisión24',
    'SURV24_aquienpideapoyo',
    'SURV24_SihayrespuestapositivaenlasdospreguntasantLasdemáspe24',
    'NEURO_DiagnosticWMlesions',
    'PEDIATRIC_AnosRepetidosCuales',
    'PEDIATRIC_AnosRepetidosRazon',
    'SURVEY24YEARS_Encasodeserpositivoespecifiquecual24',
    'SURV24_Encasodeserpositivoespecifiquecual024',
    'SURV24_Encasodeserpositivoespecifiquecualdeporte24',
    'SURV24_Encasodeserpositivoespecifiquecualhobby24',
    'SURV24_Porquesesientefelizahora24'


]

VARS_REDUNDANTES_ELIMINAR = [
    'WASI_VerbalComp',
    'WASI_PercRsng',
    'WASI_Vocabulary',
    'WASI_FSIQcat',
    'WASI_wasicat02',
    'WASI_wasimediano',
    'WASI_wasimalo',
    'WASI_FullScale2compositescore',
    'TAP_AL_EX0',
    'TAP_AL_EXT0',
    'TAP_DA3_OMT1',
    'TAP_WM2_ERT0',
    'TAP_WM2_OMT0',
    'TAP_DA3_OMT0',
    'TAP_CS_EXT1',
    'CVLT_FreeRecallIntrusions',
]

OUTCOMES_SEMAFORO = {
    'WASI_FSIQ'  : 'WASI_FullScale4compositescore',
    'CVLT_Total' : 'CVLT_Trial1a4Total',
    'VMI'        : 'VMI_estandar',
    'Fragility'  : 'RT_Fragility',
}

NOMBRES_GRUPOS = {1: 'KMC', 2: 'TC', 3: 'Referencia'}
EXCEL_ERRORS   = ['#NULL!', '#DIV/0!', '#REF!', '#VALUE!',
                  '#N/A', '#NAME?', '#NUM!']


# ══════════════════════════════════════════════════════════════════════════════
# LOGGING — fix Windows cp1252
# ══════════════════════════════════════════════════════════════════════════════

_fmt          = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
_file_handler = logging.FileHandler(LOG_FILE, mode='w', encoding='utf-8')
_file_handler.setFormatter(_fmt)
_stream       = io.TextIOWrapper(
    sys.stdout.buffer, encoding='utf-8', errors='replace', line_buffering=True
) if hasattr(sys.stdout, 'buffer') else sys.stdout
_con_handler  = logging.StreamHandler(_stream)
_con_handler.setFormatter(_fmt)
logging.basicConfig(level=logging.INFO,
                    handlers=[_file_handler, _con_handler])


# ══════════════════════════════════════════════════════════════════════════════
# UTILIDADES
# ══════════════════════════════════════════════════════════════════════════════

def to_num(s):
    return pd.to_numeric(s, errors='coerce')

def sep(title='', width=76):
    t = title.strip()
    if t:
        logging.info(f"\n{'─'*4} {t} {'─'*max(2, width-len(t)-6)}")
    else:
        logging.info('─' * width)

def tipo_variable(col, df):
    if col in FORZAR_CONTINUA:
        return 'continua_ordinal'
    v = to_num(df[col])
    n = v.nunique()
    if n > MAX_UNIQUE_CAT: return 'continua'
    if 1 < n <= MAX_UNIQUE_CAT: return 'categorica'
    return 'constante_o_vacia'

def _build_inventory(cols):
    """
    Clasifica TODAS las columnas del dataset por módulo/prefijo.
    Cualquier columna que no caiga en un módulo conocido va a 'OTROS',
    garantizando que ninguna variable se pierda del dataset maestro.
    """
    cols = list(cols)
    inv = {
        'IDs'        : [c for c in ['code','ubicac','preterm','lessthan1801g'] if c in cols],
        # 20 años — cognitivas
        'WASI'       : [c for c in cols if c.startswith('WASI_') or c == 'WASI'],
        'TAP'        : [c for c in cols if c.startswith('TAP_')  or c == 'TAP'],
        'CVLT'       : [c for c in cols if c.startswith('CVLT_') or c == 'CVLT'],
        'VMI'        : [c for c in cols if c.startswith('VMI_')  or c == 'VMI'],
        # 20 años — neuroimagen volumétrica
        'ASEG'       : [c for c in cols if c.startswith('ASEG_') or c.startswith('aseg_')
                        or c.startswith('Aseg_') or c == 'aparcvolumes'],
        'TRAC'       : [c for c in cols if c.startswith('TRAC_') or c.startswith('tracula')
                        or c == 'FA_y_MD'],
        'GMPARC'     : [c for c in cols if c.startswith('GMPARC_')],
        'APARC'      : [c for c in cols if c.startswith('APARC_') or c.startswith('aparcwm')
                        or c.startswith('aparcare') or c.startswith('aparcthi')
                        or c.startswith('aparcmea') or c == 'aparcwm' or c == 'aparcareas'
                        or c == 'aparcthickness' or c == 'aparcmeancurves'],
        'BROADMAN'   : [c for c in cols if c.startswith('BROADMAN_') or c.startswith('BROAD_')
                        or c == 'broadman'],
        'JTHP'       : [c for c in cols if c.startswith('JTHP_') or c == 'jthdescriptores'],
        'bvz'        : [c for c in cols if c.startswith('bvz_')],
        'CAM'        : [c for c in cols if c.startswith('CAM_') or c == 'CAMINO'],
        'ROI'        : [c for c in cols if c.startswith('ROI_') or c.startswith('roimarin')],
        'NEURO'      : [c for c in cols if c.startswith('NEURO_') or c.startswith('neuro_')
                        or c.startswith('Neuro_') or c == 'NEUROIMAGES'
                        or c == 'NEUROSENSORIAL' or c == 'NEUROSENS_trastornovisualoauditi'],
        # 20 años — otras evaluaciones
        'TMS20'      : [c for c in cols if c.startswith('TMS20years') or c.startswith('TMS20yea')
                        or c == 'TMS20YEARS'],
        'OPHT'       : [c for c in cols if c.startswith('OPHT_') or c == 'OPHTHALMICEVALUATION'],
        'AUDIO'      : [c for c in cols if c.startswith('AUDIO_') or c == 'AUDIOMETRICEVALUATION'],
        'CONNERS'    : [c for c in cols if c.startswith('CONNERS_') or c.startswith('connersp')
                        or c.startswith('connersj')],
        'NHPT'       : [c for c in cols if c.startswith('NHPT') and c != 'NHPT_observacionesNHPT'],
        'APEGO'      : [c for c in cols if c.startswith('APEGO_') or c == 'apego'],
        'PEDIATRIC'  : [c for c in cols if c.startswith('PEDIATRIC_') or c.startswith('PEDIATRIA_')
                        or c == 'PEDIATRICEVALUATION'],
        # 20 años — conducta / bienestar
        'ABCL'       : [c for c in cols if c.startswith('ABCL_')  or c.startswith('ASR_')
                        or c == 'ASRSELFREPORT'],
        'ABCLP'      : [c for c in cols if c.startswith('ABCLP_') or c == 'ABCLPARENTS'],
        'ABCLBF'     : [c for c in cols if c.startswith('ABCLBF_') or c == 'abclbestfriend'],
        'AUTOESTIMA' : [c for c in cols if c.startswith('AUTOESTIMA_') or c == 'autoestima'],
        'APEGO20'    : [c for c in cols if c.startswith('kidscree') or c.startswith('Kidscreen')],
        'CES_D'      : [c for c in cols if c.startswith('CES_') or c == 'cesd'],
        # 20 años — seguimiento + Fragility
        'FOLLOW'     : [c for c in cols if c.startswith('FOLLOW_') or c.startswith('RT_')],
        # 20 años — hábitos / metabólico / encuesta
        'LIFEHABITS' : [c for c in cols if c.startswith('LIFEHABITS_') or c == 'LIFEHABITS'],
        'METP'       : [c for c in cols if c.startswith('METP_') or c.startswith('PERFMET_')
                        or c == 'METABOLICPROFILE'],
        'SURV'       : [c for c in cols if c.startswith('SURV')],
        # 16-20 años — escolaridad
        'ICFES'      : [c for c in cols if c.startswith('ICFES_') or c.startswith('ICFESZ')
                        or c == 'ICFESsino' or c == 'icfesyproductivity2014'
                        or c == 'ICFES2018' or c == 'TieneencuestayICFES2018'
                        or c in ['preschoolyears_2018','ed_level_2018',
                                 'week_work_hour_2018','wages_2018','tech_stud_2018',
                                 'uni_student_2018','GradTech_2018','GradUniv_2018',
                                 'hs_graduate_2018','work_2018','sch_qua_l_2018',
                                 'lenguajesdcol_2018','lang_score_2018','math_score_2018',
                                 'social_score_2018','philos_score_2018','bio_score_2018',
                                 'chem_score_2018','physics_score_2018',
                                 'tieneiCFES2014mascompleted2018']],
        # 15 años
        'YEARS15'    : [c for c in cols if c.startswith('Years15_') or c.startswith('YEARS15_')
                        or c.startswith('TMS15_') or c == 'Chapter15YEARS'
                        or c == 'Exam15years'],
        # Primer año de vida
        'BIRTH'      : [c for c in cols if c.startswith('BIRTH_') or c == 'birth'
                        or c == 'socialcharacteristicsatbirth' or c == 'Pregnancy'
                        or c == 'neonatalstay'],
        'NEO'        : [c for c in cols if c.startswith('NEO_')],
        'SCB'        : [c for c in cols if c.startswith('SCB_') or c.startswith('CSP_')],
        'PMD'        : [c for c in cols if c.startswith('PMD_') or c.startswith('PMD12M_')
                        or c == 'PSYCHOMOTORDEVELOPMENTUP12MONTHSCA'],
        'HOMET'      : [c for c in cols if c.startswith('HOMET_') or c == 'HOME12MONTHS'
                        or c == 'HOME12_ayu_esp' or c == 'HOME'],
        'EX41'       : [c for c in cols if c.startswith('EX41_') or c == 'EXAM41WEEKS'
                        or c == 'FOLLOWUPELIGIBILITYUP41WEEKS'
                        or c.startswith('QUEST_') or c.startswith('FEEDINGS')
                        or c == 'LEARNINGSITUATION' or c == 'STRANGESITUATION_12MONTHS'],
        'FOLL12M'    : [c for c in cols if c.startswith('FOLL12M_') or c == 'HOME12_ayu_esp'],
        'NUT12M'     : [c for c in cols if c.startswith('NUT12M_') or c.startswith('Nutric')
                        or c.startswith('NUT_') or c == 'NUTRITIONUP12MONTHSCA'],
        'DOMICILIARY': [c for c in cols if c.startswith('DOMICILIARYVIST_')
                        or c == 'DOMICILIARYVISIT'],
        'LEARNINGS'  : [c for c in cols if c.startswith('LEARNINGS_')],
        'LATERALITY' : [c for c in cols if c.startswith('LAT_') or c == 'LATERALITY'
                        or c == 'SS_sit_extm'],
        # Embarazo
        'PRE'        : [c for c in cols if c.startswith('PRE_')],
        # Flags y metadatos de la base
        'METADATA'   : [c for c in cols if c in [
                        'SAMPLES','selection178ninos','Preterm178ytermRef',
                        'RCTonly','selectio','RTanalysisfactorial',
                        'NEUROIMAGES','CAMINO','roimarin','aparcwm',
                        'WMgirotemporalinferiortotal','aparcvolumes',
                        'GMgirotemporalinferiortotal_vol',
                        'filter_$','NEUROSENSORIAL','OPHTHALMICEVALUATION',
                        'AUDIOMETRICEVALUATION','PEDIATRICEVALUATION',
                        'METABOLICPROFILE','LIFEHABITS','DOMICILIARYVISIT',
                        'ABCLPARENTS','abclbestfriend','ASRSELFREPORT',
                        'connerspadres','connersjoven','apego','kidscreen',
                        'autoestima','icfesyproductivity2014','ICFESsino',
                        'TMS20YEARS','TMS20yearsSINO','ICFES2018',
                        'TieneencuestayICFES2018','Chapter15YEARS',
                        'Exam15years','EXAM41WEEKS','HOME12MONTHS',
                        'PSYCHOMOTORDEVELOPMENTUP12MONTHSCA',
                        'NUTRITIONUP12MONTHSCA','STRANGESITUATION_12MONTHS',
                        'LEARNINGSITUATION','FOLLOWUPELIGIBILITYUP41WEEKS',
                        'birth','neonatalstay','socialcharacteristicsatbirth',
                        'Pregnancy','LATERALITY','FA_y_MD','tracula','broadman',
                        'jthdescriptores','aparcwm','aparcareas','aparcthickness',
                        'aparcmeancurves','cesd','BirthDate',
                    ]],
    }

    clasificadas = set(v for lst in inv.values() for v in lst)
    inv['SIN_CLASIFICAR'] = [c for c in cols if c not in clasificadas]

    return inv

MOMENTOS = {
    'IDs':'Constante', 'WASI':'20a — cognitivo', 'TAP':'20a — cognitivo',
    'CVLT':'20a — cognitivo', 'VMI':'20a — cognitivo',
    'ASEG':'20a — neuroimagen volumétrica', 'TRAC':'20a — neuroimagen DTI',
    'GMPARC':'20a — neuroimagen parcelas corticales',
    'APARC':'20a — neuroimagen aparcellations',
    'BROADMAN':'20a — neuroimagen áreas Brodmann',
    'JTHP':'20a — neuroimagen jitter/tremor descriptors',
    'bvz':'20a — neuroimagen tractografía Bravitz',
    'CAM':'20a — neuroimagen CAM/CAMINO',
    'ROI':'20a — neuroimagen ROI',
    'NEURO':'20a — flags calidad MRI',
    'TMS20':'20a — TMS/evaluación neurológica',
    'OPHT':'20a — oftalmología',
    'AUDIO':'20a — audiometría',
    'CONNERS':'20a — Conners (atención/TDAH)',
    'NHPT':'20a — Nine Hole Peg Test',
    'APEGO':'20a — apego adulto',
    'PEDIATRIC':'20a — evaluación pediátrica',
    'ABCL':'20a — conducta adulta (ABCL/ASR)',
    'ABCLP':'20a — conducta adulta (padres)',
    'ABCLBF':'20a — conducta adulta (mejor amigo)',
    'AUTOESTIMA':'20a — autoestima',
    'APEGO20':'20a — Kidscreen / bienestar',
    'CES_D':'20a — depresión CES-D',
    'FOLLOW':'20a — seguimiento + Fragility',
    'LIFEHABITS':'20a — hábitos de vida',
    'METP':'20a — metabólico',
    'SURV':'20a — encuesta',
    'ICFES':'16-20a — escolaridad ICFES',
    'YEARS15':'15a — seguimiento intermedio',
    'BIRTH':'Nacimiento', 'NEO':'Neonatal',
    'SCB':'Nacimiento — socioeconómico',
    'PMD':'12m — desarrollo psicomotor',
    'HOMET':'12m — entorno HOME',
    'EX41':'41sem — KMC posición canguro',
    'FOLL12M':'12m — seguimiento',
    'NUT12M':'12m — nutrición',
    'DOMICILIARY':'Primer año — visita domiciliaria',
    'LEARNINGS':'Primer año — aprendizaje',
    'LATERALITY':'Evaluación — lateralidad',
    'PRE':'Embarazo — prenatal',
    'METADATA':'Metadatos de la base de datos',
    'SIN_CLASIFICAR':'Sin clasificar — revisar diccionario',
}


# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 0 — INGESTA, SANITACIÓN Y EXCLUSIÓN DE TEXTO LIBRE
# ══════════════════════════════════════════════════════════════════════════════

def etapa_0_ingesta(path):
    sep('ETAPA 0 — INGESTA, SANITACIÓN Y EXCLUSIÓN DE TEXTO LIBRE')

    df    = pd.read_excel(path)
    n_err = sum((df == e).sum().sum() for e in EXCEL_ERRORS)
    n_nan = df.isna().sum().sum()
    logging.info(f"Archivo       : {path}")
    logging.info(f"Dimensiones   : {df.shape[0]} × {df.shape[1]}")
    logging.info(f"Errores Excel : {n_err:>10,}")
    logging.info(f"NaN reales    : {n_nan:>10,}")
    logging.info(f"Total problema: {n_err+n_nan:>10,}  ({(n_err+n_nan)/df.size*100:.1f}%)")
    df = df.replace(EXCEL_ERRORS, np.nan)
    logging.info("Errores → NaN ✓")

    sep('0A — Exclusión de variables de texto libre')
    logging.info("Criterio: texto narrativo → no modelable. "
                 "Contenido clínico capturado en variables codificadas.\n")
    excluidas = []
    for col in VARS_TEXTO_LIBRE:
        if col in df.columns:
            pct = df[col].isna().mean() * 100
            logging.info(f"  ✗  {col:<52}  {pct:>5.1f}% NaN  n={int(df[col].notna().sum())}")
            excluidas.append(col)
        else:
            logging.info(f"  —  {col:<52}  (no presente)")
    df = df.drop(columns=excluidas, errors='ignore')
    logging.info(f"\n  Excluidas: {len(excluidas)} | Dataset: {df.shape[0]} × {df.shape[1]}")

    inventory = _build_inventory(df.columns)

    sep('Inventario de módulos por momento temporal')
    logging.info(f"  {'Módulo':<12} {'Vars':>6}  {'Momento temporal'}")
    logging.info('  ' + '─' * 65)
    total = 0
    for mod, lst in inventory.items():
        if lst:
            logging.info(f"  {mod:<12} {len(lst):>6}  {MOMENTOS.get(mod,'')}")
            total += len(lst)
    clasificadas = set(v for lst in inventory.values() for v in lst)
    otras = [c for c in df.columns if c not in clasificadas]
    logging.info(f"\n  Clasificadas    : {total}")
    logging.info(f"  Sin clasificar  : {len(otras)}")
    logging.info(f"  Total columnas  : {len(df.columns)}")

    outcomes_core = (inventory['WASI'] + inventory['TAP'] +
                     inventory['CVLT'] + inventory['VMI'])
    grupos = to_num(df['ubicac'])
    sep('Distribución de grupos y atricición')
    for g, nombre in NOMBRES_GRUPOS.items():
        logging.info(f"  {nombre:<12}: n={(grupos==g).sum()}")
    n_ini = len(df)
    df    = df.dropna(subset=[c for c in outcomes_core if c in df.columns],
                      how='all').copy()
    n_ret = len(df)
    logging.info(f"\n  Atricición: {n_ini} → {n_ret} ({n_ini-n_ret} excluidos)")

    return df, inventory


# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 2 — ANÁLISIS DE MISSING DATA
# ══════════════════════════════════════════════════════════════════════════════

def etapa_2_missing(df, inventory):
    sep('ETAPA 2 — ANÁLISIS DE MISSING DATA')

    grupos     = to_num(df['ubicac'])
    todas_cols = [c for mod in inventory for c in inventory[mod]
                  if c in df.columns and mod != 'IDs']

    # Series de outcomes para semáforo
    out_series = {}
    for nombre, col in OUTCOMES_SEMAFORO.items():
        if col in df.columns:
            s = to_num(df[col])
            if s.notna().sum() >= MIN_DATOS_CORR:
                out_series[nombre] = s
    out_nombres = list(out_series.keys())

    # 2A — Cat.1: Completamente vacías
    sep('2A — Cat.1: Completamente vacías (100% NaN)')
    vacias = [c for c in todas_cols if df[c].isna().mean() >= 1.0]
    logging.info(f"  Total: {len(vacias)} variables")
    for col in sorted(vacias):
        logging.info(f"  ✗  {col}")

    # 2B — Cat.2: Alta ausencia + semáforo
    sep('2B — Cat.2: Alta ausencia (>70% NaN) — semáforo Spearman')
    logging.info("FILOSOFÍA: alta ausencia ≠ mala variable.\n")

    alta_ausencia = [c for c in todas_cols if 0.70 <= df[c].isna().mean() < 1.0]
    semaforos     = {}
    res_alta      = []

    header = ''.join([f"{n[:8]:>10}" for n in out_nombres])
    logging.info(f"  {'Variable':<52} {'%NaN':>6} {'N':>5}  {header}  Semáforo")
    logging.info('  ' + '─' * (70 + 10*len(out_nombres)))

    for col in sorted(alta_ausencia, key=lambda c: -df[c].isna().mean()):
        pct = df[col].isna().mean() * 100
        n   = int(df[col].notna().sum())
        v   = to_num(df[col])
        corrs = {}
        for nombre, out_s in out_series.items():
            mask = v.notna() & out_s.notna()
            if mask.sum() >= MIN_DATOS_CORR:
                r, p = spearmanr(v[mask], out_s[mask])
                corrs[nombre] = {'r': round(float(r), 3),
                                 'sig': abs(r) >= OUTCOME_THRESH and p < 0.05}
            else:
                corrs[nombre] = {'r': None, 'sig': False}
        n_sig = sum(1 for c in corrs.values() if c['sig'])
        sem   = ('VERDE' if n_sig >= 2 else 'AMARILLO' if n_sig == 1 else
                 'GRIS'  if n < MIN_DATOS_CORR else 'ROJO')
        semaforos[col] = sem
        r_vals = ''.join([
            f"{corrs[nb]['r']:>10.3f}" if corrs[nb]['r'] is not None
            else f"{'n/a':>10}" for nb in out_nombres
        ])
        logging.info(f"  {col:<52} {pct:>5.1f}% {n:>5}  {r_vals}  {sem}")
        res_alta.append({'variable': col, 'pct_nan': pct, 'n': n,
                         'semaforo': sem, 'n_sig': n_sig,
                         **{f'r_{nb}': corrs[nb]['r'] for nb in out_nombres}})

    from collections import Counter
    cnt = Counter(r['semaforo'] for r in res_alta)
    sep('Resumen semáforo Cat.2')
    logging.info(f"  VERDE    ({cnt.get('VERDE',0):>3}) correlaciona ≥2 outcomes → NO eliminar")
    logging.info(f"  AMARILLO ({cnt.get('AMARILLO',0):>3}) correlaciona 1 outcome  → investigar")
    logging.info(f"  GRIS     ({cnt.get('GRIS',0):>3}) pocos datos              → revisar")
    logging.info(f"  ROJO     ({cnt.get('ROJO',0):>3}) sin correlación relevante → candidata descartar")

    # 2C — Cat.3: Redundantes
    sep('2C — Cat.3: Variables redundantes (|r| > 0.90)')
    logging.info("Dataset maestro las CONSERVA. Dataset de modelado las excluye.\n")
    TABLA = [
        ('WASI_VerbalComp',               'WASI_VerbalComprehcompositescore',    0.999, 'compositescore',  'VerbalComp'),
        ('WASI_PercRsng',                 'WASI_PercRsngcompositescore',          0.999, 'compositescore',  'PercRsng'),
        ('TAP_AL_EX0',                    'TAP_AL_EXT0',                          0.985, 'AL_EXT0_CLIN',    'ambas crudas'),
        ('WASI_FSIQcat',                  'WASI_wasicat02',                       0.967, 'ninguna (cat.)',   'ambas categ.'),
        ('TAP_DA3_OMT1',                  'TAP_DA3_OMT1_CLIN',                    0.954, 'DA3_OMT1_CLIN',   'DA3_OMT1'),
        ('WASI_Vocabulary',               'WASI_VerbalComprehcompositescore',    0.945, 'compositescore',  'Vocabulary'),
        ('WASI_FullScale4compositescore', 'WASI_FSIQcat',                         0.945, 'FullScale4',      'FSIQcat'),
        ('WASI_Vocabulary',               'WASI_VerbalComp',                      0.943, 'compositescore',  'Vocabulary'),
        ('TAP_WM2_ERT0',                  'TAP_WM2_ERT0_CLIN',                    0.934, 'WM2_ERT0_CLIN',   'WM2_ERT0'),
        ('CVLT_FreeRecallIntrusions',     'CVLT_TotalIntrusions',                 0.931, 'TotalIntrusions', 'FreeRecall'),
        ('WASI_FullScale4compositescore', 'WASI_FullScale2compositescore',        0.929, 'FullScale4',      'FullScale2'),
        ('TAP_WM2_OMT0',                  'TAP_WM2_OMT0_CLIN',                    0.921, 'WM2_OMT0_CLIN',   'WM2_OMT0'),
        ('WASI_FullScale4compositescore', 'WASI_wasicat02',                       0.914, 'FullScale4',      'wasicat02'),
        ('TAP_DA3_OMT0',                  'TAP_DA3_OMT2',                         0.907, 'OMT2',            'OMT0'),
        ('TAP_CS_EXT1',                   'TAP_CS_EXT1_CLIN',                     0.904, 'CS_EXT1_CLIN',    'CS_EXT1'),
    ]
    logging.info(f"  {'Variable A':<45} {'Variable B':<45} {'r':>6}  Conservar → Eliminar")
    logging.info('  ' + '─' * 110)
    for v1, v2, r, cons, elim in TABLA:
        logging.info(f"  {v1:<45} {v2:<45} {r:>6.3f}  {cons} → {elim}")
    logging.info(f"\n  Variables a excluir en modelado ({len(VARS_REDUNDANTES_ELIMINAR)}):")
    for v in VARS_REDUNDANTES_ELIMINAR:
        logging.info(f"    ✗  {v}")

    # 2D — Chi-cuadrado por bloque temporal
    sep('2D — Chi-cuadrado MCAR vs MAR por bloque temporal')
    for bloque, mods in [
        ('20a_cognitivas', ['WASI','TAP','CVLT','VMI']),
        ('20a_otras',      ['ICFES','ABCL','FOLLOW','LIFEHABITS']),
        ('primer_año',     ['BIRTH','NEO','SCB','PMD','HOMET']),
    ]:
        cols_b = [c for mod in mods
                  for c in inventory.get(mod, []) if c in df.columns]
        if not cols_b:
            continue
        n_mcar = n_mar = 0
        sep(f'Chi-cuadrado — {bloque}')
        logging.info(
            f"  {'Variable':<50} {'Chi2':>7}  {'p-val':>8}  "
            f"{'KMC%':>6} {'TC%':>6} {'Ref%':>6}  Mecanismo"
        )
        logging.info('  ' + '─' * 100)
        for col in cols_b:
            v     = to_num(df[col])
            tiene = v.notna().astype(int)
            sub   = pd.DataFrame({'g': grupos, 't': tiene})
            sub   = sub[sub['g'].isin([1, 2, 3])]
            ct    = pd.crosstab(sub['g'], sub['t'])
            if ct.shape != (3, 2):
                continue
            try:
                chi2_v, p_v, _, _ = chi2_contingency(ct)
                n_g  = {g: (grupos==g).sum() for g in [1,2,3]}
                pcts = {g: round(v[grupos==g].notna().sum()/n_g[g]*100,1)
                        for g in [1,2,3]}
                delta = round(max(pcts.values())-min(pcts.values()), 1)
                mec   = 'MCAR' if p_v >= MISSING_DIFF_P \
                        else f'MAR ⚠ (Δ={delta}pp)'
                n_mcar += p_v >= MISSING_DIFF_P
                n_mar  += p_v < MISSING_DIFF_P
                logging.info(
                    f"  {col:<50} {chi2_v:>7.2f}  {p_v:>8.4f}  "
                    f"{pcts[1]:>6.1f} {pcts[2]:>6.1f} {pcts[3]:>6.1f}  {mec}"
                )
            except Exception:
                pass
        logging.info(f"\n  [{bloque}] MCAR={n_mcar} | MAR={n_mar}")

    return vacias, semaforos


# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 3 — PREPROCESAMIENTO QC
# ══════════════════════════════════════════════════════════════════════════════

def etapa_3_preprocesamiento(df, inventory):
    sep('ETAPA 3 — PREPROCESAMIENTO QC')

    sep(f'3A — Rangos clínicos WASI {WASI_CLINICAL_RANGE} → fuera de rango = NaN')
    n_fuera = 0
    for col in inventory.get('WASI', []):
        if col not in df.columns: continue
        df[col] = to_num(df[col])
        mask    = ((df[col] < WASI_CLINICAL_RANGE[0]) |
                   (df[col] > WASI_CLINICAL_RANGE[1]))
        n_col   = mask.sum()
        n_fuera += n_col
        if n_col > 0:
            logging.info(f"  {col}: {n_col} valores fuera de rango → NaN")
        df.loc[mask, col] = np.nan
    logging.info(f"  Total corregidos: {n_fuera}")

    sep(f'3B — Auditoría IQR Pasiva ({IQR_MULT}×IQR — ningún dato eliminado)')
    logging.info(f"  {'Variable':<52} {'N outliers':>11}  {'Q1':>8} {'Q3':>8}")
    logging.info('  ' + '─' * 85)
    n_alertas = 0
    for mod in ['TAP','CVLT','VMI','BIRTH','NEO','PMD','HOMET','FOLL12M']:
        for col in inventory.get(mod, []):
            if col not in df.columns: continue
            df[col] = to_num(df[col])
            q1, q3  = df[col].quantile(0.25), df[col].quantile(0.75)
            iqr     = q3 - q1
            if iqr == 0: continue
            n_out = int(((df[col] < q1-IQR_MULT*iqr) |
                         (df[col] > q3+IQR_MULT*iqr)).sum())
            if n_out > 0:
                logging.info(f"  {col:<52} {n_out:>11}  {q1:>8.2f} {q3:>8.2f}")
                n_alertas += 1
    logging.info(f"  Variables con alertas: {n_alertas} (datos preservados)")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 4 — INGENIERÍA DE FEATURES DE NEUROIMAGEN
# ══════════════════════════════════════════════════════════════════════════════

def etapa_4_neuroimagen(df, inventory):
    sep('ETAPA 4 — INGENIERÍA DE FEATURES DE NEUROIMAGEN')

    for flag in MRI_QUALITY_FLAGS:
        if flag in df.columns:
            df[flag] = to_num(df[flag])

    if all(f in df.columns for f in MRI_QUALITY_FLAGS):
        df['has_MRI'] = ((df[MRI_QUALITY_FLAGS[0]] == 1) |
                         (df[MRI_QUALITY_FLAGS[1]] == 1)).astype(int)
        logging.info(f"has_MRI: n={df['has_MRI'].sum()} (T1 OK o DTI OK)")
    elif ICV_COLUMN in df.columns:
        df['has_MRI'] = to_num(df[ICV_COLUMN]).notna().astype(int)
        logging.warning("Flags MRI no encontrados — usando ICV como fallback")
    else:
        df['has_MRI'] = 0

    n_norm = 0
    if ICV_COLUMN in df.columns:
        df[ICV_COLUMN] = to_num(df[ICV_COLUMN])
        for col in [c for c in inventory.get('ASEG', [])
                    if c != ICV_COLUMN and c in df.columns]:
            df[f'n_{col}'] = to_num(df[col]) / df[ICV_COLUMN]
            n_norm += 1
    logging.info(f"n_ASEG_ generadas: {n_norm} variables")

    viq = 'WASI_VerbalComprehcompositescore'
    piq = 'WASI_PercRsngcompositescore'
    if viq in df.columns and piq in df.columns:
        df['DESC_WASI_VIQ_PIQ_gap'] = (to_num(df[viq]) - to_num(df[piq])).abs()
        logging.info("DESC_WASI_VIQ_PIQ_gap creada (solo descriptiva)")

    n_aseg_cols = [c for c in df.columns if c.startswith('n_ASEG_')]
    return df, n_aseg_cols


# ══════════════════════════════════════════════════════════════════════════════
# ETAPA 5 — AUDITORÍA FINAL
# ══════════════════════════════════════════════════════════════════════════════

def etapa_5_auditoria(df, inventory, n_aseg_cols):
    sep('ETAPA 5 — AUDITORÍA FINAL')

    cogn_cols = [c for mod in ['WASI','TAP','CVLT','VMI']
                 for c in inventory.get(mod, []) if c in df.columns]
    df_out    = df[cogn_cols].apply(to_num)
    corr      = df_out.corr().abs()
    pares     = [(corr.columns[i], corr.columns[j], corr.iloc[i,j])
                 for i in range(len(corr.columns))
                 for j in range(i)
                 if corr.iloc[i,j] > CORR_THRESHOLD]

    sep('5A — Verificación de redundancia (r > 0.90) en cognitivas')
    logging.info(f"  Pares detectados: {len(pares)}")
    for p1, p2, r in sorted(pares, key=lambda x: -x[2]):
        en_lista = p1 in VARS_REDUNDANTES_ELIMINAR or p2 in VARS_REDUNDANTES_ELIMINAR
        logging.info(f"  r={r:.3f}  {p1}  vs  {p2}  "
                     f"→ {'✓ en lista' if en_lista else '⚠ NO en lista — revisar'}")

    sep('5B — Auditoría de sesgo socioeconómico')
    logging.info(f"  {'Variable':<40} {'p-valor':>8}  Resultado")
    logging.info('  ' + '─' * 55)
    bias_vars = {'Genero': 'BIRTH_sexo5'}
    for col in inventory.get('SCB', []):
        if col in df.columns:
            df[col] = to_num(df[col])
            if 1 < df[col].nunique() <= 10:
                bias_vars[f'SCB_{col}'] = col
    grupos = to_num(df['ubicac'])
    for label, col in bias_vars.items():
        if col not in df.columns: continue
        tmp = df.dropna(subset=['ubicac', col])
        ct  = pd.crosstab(to_num(tmp['ubicac']), to_num(tmp[col]))
        try:
            _, p, _, _ = chi2_contingency(ct)
            logging.info(f"  {label:<40} {p:>8.4f}  "
                         f"{'PASS ✓' if p >= 0.05 else 'ALERTA ⚠'}")
        except Exception:
            pass

    return df


# ══════════════════════════════════════════════════════════════════════════════
# EXPORT — DATASETS
# ══════════════════════════════════════════════════════════════════════════════

def exportar_datasets(df, inventory, n_aseg_cols):
    sep('EXPORT — DATASETS MAESTRO + MODELADO + INTERIM')

    id_cols = [c for c in ['code','ubicac','preterm','lessthan1801g','has_MRI']
               if c in df.columns]

    # ── Módulos por bloque temporal ───────────────────────────────────────
    BLOQUES = {
        '20a_cognitivas' : ['WASI','TAP','CVLT','VMI'],
        '20a_neuro_vol'  : ['ASEG','GMPARC','APARC','BROADMAN','JTHP','bvz',
                            'CAM','ROI'],
        '20a_neuro_dti'  : ['TRAC'],
        '20a_neuro_flags': ['NEURO'],
        '20a_otras_eval' : ['TMS20','OPHT','AUDIO','CONNERS','NHPT','APEGO',
                            'PEDIATRIC'],
        '20a_conducta'   : ['ABCL','ABCLP','ABCLBF','AUTOESTIMA','APEGO20',
                            'CES_D'],
        '20a_seguimiento': ['FOLLOW'],
        '20a_habitos'    : ['LIFEHABITS','METP','SURV'],
        '16_20a_escolar' : ['ICFES'],
        '15a'            : ['YEARS15'],
        'primer_ano'     : ['BIRTH','NEO','SCB','PMD','HOMET','EX41',
                            'FOLL12M','NUT12M','DOMICILIARY','LEARNINGS',
                            'LATERALITY'],
        'embarazo'       : ['PRE'],
        'metadata'       : ['METADATA','SIN_CLASIFICAR'],
    }

    # Construir lista ordenada sin duplicados
    cols_bloque = []
    for mods in BLOQUES.values():
        for mod in mods:
            cols_bloque += [c for c in inventory.get(mod, [])
                            if c in df.columns]

    # n_ASEG_ generadas por pipeline (no están en inventory original)
    cols_naseg = [c for c in n_aseg_cols if c in df.columns]
    cols_desc  = [c for c in df.columns if c.startswith('DESC_')]

    # Garantía final: cualquier columna del df que no haya caído arriba
    incluidas  = set(id_cols + cols_bloque + cols_naseg + cols_desc)
    cols_extra = [c for c in df.columns if c not in incluidas]

    cols_maestro = list(dict.fromkeys(
        id_cols + cols_bloque + cols_naseg + cols_desc + cols_extra
    ))

    # Verificar cobertura total
    n_perdidas = len([c for c in df.columns if c not in cols_maestro])
    assert n_perdidas == 0, \
        f"BUG: {n_perdidas} columnas del df no están en cols_maestro"

    cols_modelo = [c for c in cols_maestro if c not in VARS_REDUNDANTES_ELIMINAR]

    df[cols_maestro].to_csv(OUT_MASTER,  index=False)
    df[cols_modelo].to_csv(OUT_MODELO,   index=False)
    df[cols_maestro].to_csv(OUT_INTERIM, index=False)

    n_red = len([c for c in VARS_REDUNDANTES_ELIMINAR if c in cols_maestro])

    # Desglose por bloque
    logging.info(f"\n  Desglose de columnas en dataset maestro:")
    logging.info(f"  {'Bloque':<25} {'Módulos':<45} {'N vars':>7}")
    logging.info('  ' + '─' * 80)
    for bloque, mods in BLOQUES.items():
        n = sum(len([c for c in inventory.get(m,[]) if c in df.columns]) for m in mods)
        if n > 0:
            logging.info(f"  {bloque:<25} {', '.join(mods):<45} {n:>7}")
    logging.info(f"  {'n_ASEG_ (norm. ICV)':<25} {'—':<45} {len(cols_naseg):>7}")
    logging.info(f"  {'DESC_ (derivadas)':<25} {'—':<45} {len(cols_desc):>7}")
    logging.info(f"  {'Extra (garantía)':<25} {'—':<45} {len(cols_extra):>7}")

    logging.info(f"\n  {OUT_INTERIM:<42} → input para v9a_eda.py")
    logging.info(f"  {OUT_MASTER:<42} → {len(cols_maestro)} vars (trazabilidad)")
    logging.info(f"  {OUT_MODELO:<42} → {len(cols_modelo)} vars "
                 f"(sin {n_red} redundantes)")
    logging.info(f"\n  Participantes                : {len(df)}")
    logging.info(f"  Vars Excel original          : {df.shape[1]}")
    logging.info(f"  Vars dataset maestro         : {len(cols_maestro)}")
    logging.info(f"  Cobertura                    : "
                 f"{len(cols_maestro)/df.shape[1]*100:.1f}%")

    return cols_maestro, cols_modelo


# ══════════════════════════════════════════════════════════════════════════════
# EXPORT — DICCIONARIO DE TIPOS
# ══════════════════════════════════════════════════════════════════════════════

def exportar_diccionario_tipos(df, inventory, cols_modelo_set):
    sep('EXPORT — DICCIONARIO DE TIPOS DE VARIABLE')

    mod_de_col = {}
    mom_de_col = {}
    for mod, lst in inventory.items():
        for col in lst:
            mod_de_col[col] = mod
            mom_de_col[col] = MOMENTOS.get(mod, '')
    for col in df.columns:
        if col not in mod_de_col:
            if col.startswith('n_ASEG_'):
                mod_de_col[col] = 'ASEG_norm'
                mom_de_col[col] = '20a — neuroimagen normalizada'
            elif col.startswith('DESC_'):
                mod_de_col[col] = 'DERIVED'
                mom_de_col[col] = 'Derivada'
            elif col == 'has_MRI':
                mod_de_col[col] = 'IDs'
                mom_de_col[col] = 'Constante'
            else:
                mod_de_col[col] = 'OTRO'
                mom_de_col[col] = ''

    ID_COLS = {'code', 'ubicac', 'preterm', 'lessthan1801g', 'has_MRI'}
    filas   = []
    for col in df.columns:
        v   = to_num(df[col])
        pct = df[col].isna().mean() * 100
        n   = int(df[col].notna().sum())
        n_u = int(v.nunique())

        if col in ID_COLS:            tipo = 'ID'
        elif col in FORZAR_CONTINUA:  tipo = 'Ordinal_1_3'
        elif n_u == 0:                tipo = 'Vacía'
        elif n_u == 1:                tipo = 'Constante'
        elif n_u == 2:                tipo = 'Binaria'
        elif n_u <= MAX_UNIQUE_CAT:   tipo = 'Categórica'
        else:                         tipo = 'Numérica'

        vals_unicos = n_cats = ''
        if tipo in ('Binaria', 'Categórica', 'Ordinal_1_3', 'ID'):
            uniq       = sorted(v.dropna().unique().tolist())
            n_cats     = str(n_u)
            vals_unicos = str(uniq[:15])

        en_modelo = 'Sí' if col in cols_modelo_set else 'No'
        razon     = ''
        if en_modelo == 'No':
            if col in VARS_TEXTO_LIBRE:           razon = 'Texto libre'
            elif col in VARS_REDUNDANTES_ELIMINAR: razon = 'Redundante r>0.90'
            elif col in ID_COLS:                  razon = 'Identificador'
            else:                                 razon = 'No en dataset modelado'

        filas.append({
            'variable': col, 'modulo': mod_de_col.get(col,''),
            'momento_temporal': mom_de_col.get(col,''),
            'tipo_datos': tipo, 'n_categorias': n_cats,
            'valores_unicos': vals_unicos,
            'pct_nan': round(pct,1), 'n_datos': n,
            'en_modelado': en_modelo, 'razon_exclusion': razon,
        })

    ORDEN_TIPO = {'ID':0,'Numérica':1,'Ordinal_1_3':2,
                  'Binaria':3,'Categórica':4,'Constante':5,'Vacía':6}
    df_tipos = pd.DataFrame(filas)
    df_tipos['_o'] = df_tipos['tipo_datos'].map(lambda t: ORDEN_TIPO.get(t,9))
    df_tipos = df_tipos.sort_values(['modulo','_o','variable']).drop(columns=['_o'])
    df_tipos.to_csv(OUT_TIPOS, index=False, encoding='utf-8-sig')

    resumen = df_tipos['tipo_datos'].value_counts()
    logging.info(f"\n  {'Tipo':<20} {'N':>6}")
    logging.info('  ' + '─' * 28)
    for t, c in resumen.items():
        logging.info(f"  {t:<20} {c:>6}")
    logging.info(f"\n  → {OUT_TIPOS}  ({len(df_tipos)} variables)")
    return df_tipos


# ══════════════════════════════════════════════════════════════════════════════
# EXPORT — DICCIONARIO MISSING + REPORTE + REVISIÓN
# ══════════════════════════════════════════════════════════════════════════════

def exportar_diccionario_missing(df, inventory, vacias, semaforos):
    sep('EXPORT — DICCIONARIO MISSING + REPORTE + REVISIÓN')

    grupos = to_num(df['ubicac'])
    n_grp  = {g: (grupos==g).sum() for g in [1,2,3]}
    MNAR_MODULOS  = {'ASEG', 'TRAC', 'ASEG_norm'}
    MNAR_BENIGNO  = {'NEURO'}

    mod_de_col = {}
    mom_de_col = {}
    for mod, lst in inventory.items():
        for col in lst:
            mod_de_col[col] = mod
            mom_de_col[col] = MOMENTOS.get(mod, '')
    for col in df.columns:
        if col not in mod_de_col:
            if col.startswith('n_ASEG_'):   mod_de_col[col] = 'ASEG_norm'
            elif col.startswith('DESC_'):   mod_de_col[col] = 'DERIVED'
            elif col == 'has_MRI':          mod_de_col[col] = 'IDs'
            else:                           mod_de_col[col] = 'OTRO'
            mom_de_col[col] = ''

    ID_COLS = {'code','ubicac','preterm','lessthan1801g','has_MRI'}
    filas   = []
    for col in df.columns:
        v   = to_num(df[col])
        pct = df[col].isna().mean() * 100
        n   = int(df[col].notna().sum())
        mod = mod_de_col.get(col, 'OTRO')

        pct_g = {g: round(v[grupos==g].notna().sum()/n_grp.get(g,1)*100, 1)
                 for g in [1,2,3]}

        tiene   = v.notna().astype(int)
        sub     = pd.DataFrame({'g':grupos,'t':tiene})
        sub     = sub[sub['g'].isin([1,2,3])]
        ct      = pd.crosstab(sub['g'], sub['t'])
        chi2_s, chi2_p = None, None
        if ct.shape == (3, 2):
            try:
                chi2_s, chi2_p, _, _ = chi2_contingency(ct)
            except Exception:
                pass

        if col in VARS_TEXTO_LIBRE:          mec = 'Texto_libre'
        elif col in ID_COLS:                 mec = 'ID'
        elif pct == 0.0:                     mec = 'Completa'
        elif pct >= 99.9:                    mec = 'Vacía_100pct'
        elif mod in MNAR_MODULOS or col.startswith('ASEG_') or col.startswith('TRAC_'):
            mec = 'MNAR'
        elif mod in MNAR_BENIGNO:            mec = 'MNAR_benigno'
        elif chi2_p is not None and chi2_p < MISSING_DIFF_P:  mec = 'MAR'
        elif chi2_p is not None:             mec = 'MCAR'
        else:                                mec = 'Desconocido'

        sem = semaforos.get(col, '') if pct >= 70 else ''

        # Estrategia de imputación
        if mec in ('ID','Texto_libre'):
            est, just = 'NA', 'No es una variable analítica'
        elif mec == 'Completa':
            est, just = 'NA', 'Sin faltantes — no requiere imputación'
        elif mec == 'Vacía_100pct':
            est, just = 'Eliminar', 'Sin datos — eliminar del dataset'
        elif mec in ('MNAR','MNAR_benigno'):
            est, just = 'Submuestra', \
                ('Ausencia por protocolo MRI. Usar submuestra n≈299. '
                 'NO imputar — sesgo garantizado.')
        elif mec == 'MAR':
            delta = round(max(pct_g.values())-min(pct_g.values()), 1)
            if delta >= 15:
                est, just = 'No_imputar_excluir', \
                    (f'MAR severo (Δ={delta}pp). Usar solo KMC vs TC. '
                     'NO incluir en clustering de 3 grupos.')
            else:
                est, just = 'Mediana_CV', \
                    (f'MAR leve (Δ={delta}pp). Imputar mediana en train dentro de CV.')
        elif mec == 'MCAR':
            if pct <= 5:
                est, just = 'Mediana_CV', f'MCAR, bajo ({pct:.1f}%). Imputar mediana en CV.'
            elif pct <= 30:
                est, just = 'Mediana_CV', f'MCAR, moderado ({pct:.1f}%). Mediana en CV.'
            elif pct < 70:
                est, just = 'Revisar_experto', \
                    f'MCAR, alto ({pct:.1f}%). Revisar diccionario antes de imputar.'
            else:
                if sem == 'VERDE':
                    est, just = 'Revisar_experto', \
                        f'MCAR {pct:.1f}% VERDE: correlaciona ≥2 outcomes. NO eliminar.'
                elif sem == 'AMARILLO':
                    est, just = 'Revisar_experto', \
                        f'MCAR {pct:.1f}% AMARILLO: correlaciona 1 outcome. Investigar.'
                else:
                    est, just = 'Candidata_descartar', \
                        f'MCAR {pct:.1f}% {sem}: sin correlación relevante. Consultar experto.'
        else:
            est, just = 'Revisar_experto', 'Mecanismo no determinado'

        filas.append({
            'variable': col, 'modulo': mod,
            'momento_temporal': mom_de_col.get(col,''),
            'pct_nan': round(pct,1), 'n_datos': n,
            'mecanismo': mec,
            'chi2_stat': round(float(chi2_s),3) if chi2_s is not None else '',
            'chi2_p':    round(float(chi2_p),4) if chi2_p is not None else '',
            'pct_kmc': pct_g[1], 'pct_tc': pct_g[2], 'pct_ref': pct_g[3],
            'semaforo_cat2': sem,
            'estrategia_imput': est, 'justificacion': just,
            'decision_final': '',
        })

    ORDEN_MEC = {'MNAR':0,'MNAR_benigno':1,'MAR':2,'MCAR':3,
                 'Completa':4,'ID':5,'Texto_libre':6,'Vacía_100pct':7,'Desconocido':8}
    ORDEN_SEM = {'VERDE':0,'AMARILLO':1,'GRIS':2,'ROJO':3,'':4}

    df_missing = pd.DataFrame(filas)
    df_missing['_om'] = df_missing['mecanismo'].map(lambda m: ORDEN_MEC.get(m,9))
    df_missing = df_missing.sort_values(
        ['_om','pct_nan'], ascending=[True,False]).drop(columns=['_om'])
    df_missing.to_csv(OUT_MISSING, index=False, encoding='utf-8-sig')

    # ── missing_reporte.csv — tabla completa ordenada por urgencia ────────
    df_reporte = df_missing.copy()
    df_reporte['_om'] = df_reporte['mecanismo'].map(lambda m: ORDEN_MEC.get(m,9))
    df_reporte['_os'] = df_reporte['semaforo_cat2'].map(lambda s: ORDEN_SEM.get(s,4))
    df_reporte = df_reporte.sort_values(
        ['_om','_os','pct_nan'], ascending=[True,True,False]
    ).drop(columns=['_om','_os'])
    df_reporte.to_csv(OUT_REPORTE, index=False, encoding='utf-8-sig')

    # ── missing_para_revision.csv — solo las que requieren decisión humana ─
    ESTRATEGIAS_REVISION = {'Revisar_experto','Candidata_descartar',
                             'No_imputar_excluir','Eliminar'}
    MECANISMOS_REVISION  = {'Vacía_100pct','MAR','Desconocido'}
    mask = (
        df_missing['estrategia_imput'].isin(ESTRATEGIAS_REVISION) |
        (df_missing['mecanismo'].isin(MECANISMOS_REVISION) &
         (df_missing['mecanismo'] != 'Completa')) |
        (df_missing['pct_nan'] >= 70)
    )
    df_rev = df_missing[mask].copy()
    df_rev['definicion_diccionario'] = ''
    df_rev['consulta_experto']       = ''
    df_rev['decision_final_equipo']  = ''
    df_rev['_os'] = df_rev['semaforo_cat2'].map(lambda s: ORDEN_SEM.get(s,4))
    df_rev['_om'] = df_rev['mecanismo'].map(lambda m: ORDEN_MEC.get(m,9))
    df_rev = df_rev.sort_values(
        ['_os','_om','pct_nan'], ascending=[True,True,False]
    ).drop(columns=['_os','_om'])
    df_rev.to_csv(OUT_REVISION, index=False, encoding='utf-8-sig')

    # Resumen en log
    sep('Resumen estrategias')
    for label, series in [('Mecanismo','mecanismo'),('Estrategia','estrategia_imput')]:
        resumen = df_missing[series].value_counts()
        logging.info(f"\n  {label}:")
        for k, c in resumen.items():
            logging.info(f"    {k:<30} {c:>5}")

    n_verde    = (df_rev['semaforo_cat2']=='VERDE').sum()
    n_amarillo = (df_rev['semaforo_cat2']=='AMARILLO').sum()
    n_mar_rev  = (df_rev['mecanismo']=='MAR').sum()
    n_vacias   = (df_rev['mecanismo']=='Vacía_100pct').sum()
    logging.info(f"\n  {OUT_MISSING:<42} → {len(df_missing)} variables")
    logging.info(f"  {OUT_REPORTE:<42} → {len(df_reporte)} variables (tabla completa)")
    logging.info(f"  {OUT_REVISION:<42} → {len(df_rev)} para revisión experta")
    logging.info(f"\n  Desglose revisión:")
    logging.info(f"    VERDE    (NO eliminar)  : {n_verde:>4}")
    logging.info(f"    AMARILLO (investigar)   : {n_amarillo:>4}")
    logging.info(f"    MAR diferencial         : {n_mar_rev:>4}")
    logging.info(f"    Vacías 100%             : {n_vacias:>4}")
    logging.info(f"\n  Instrucciones para revisión:")
    logging.info(f"    1. Ctrl-F cada variable en el diccionario PDF")
    logging.info(f"    2. Priorizar VERDE y AMARILLO primero")
    logging.info(f"    3. Rellenar: definicion_diccionario, consulta_experto, decision_final_equipo")

    return df_missing


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main():
    df, inventory             = etapa_0_ingesta(INPUT_FILE)
    vacias, semaforos         = etapa_2_missing(df, inventory)
    df                        = etapa_3_preprocesamiento(df, inventory)
    df, n_aseg_cols           = etapa_4_neuroimagen(df, inventory)
    df                        = etapa_5_auditoria(df, inventory, n_aseg_cols)
    cols_maestro, cols_modelo = exportar_datasets(df, inventory, n_aseg_cols)
    cols_modelo_set           = set(cols_modelo)
    df_tipos                  = exportar_diccionario_tipos(df, inventory, cols_modelo_set)
    df_missing                = exportar_diccionario_missing(df, inventory, vacias, semaforos)

    sep('RESUMEN FINAL — v9b')
    outputs = [
        (OUT_INTERIM,  f"Input para v9a_eda.py         ({len(cols_maestro)} vars)"),
        (OUT_MASTER,   f"Dataset maestro completo      ({len(cols_maestro)} vars)"),
        (OUT_MODELO,   f"Dataset de modelado           ({len(cols_modelo)} vars)"),
        (OUT_TIPOS,    f"Diccionario tipos             ({len(df_tipos)} vars)"),
        (OUT_MISSING,  f"Diccionario missing           ({len(df_missing)} vars)"),
        (OUT_REPORTE,  f"Missing reporte completo      ({len(df_missing)} vars)"),
        (OUT_REVISION, f"Missing para revisión         (ver log para desglose)"),
        (LOG_FILE,     "Log completo"),
    ]
    logging.info(f"\n  {'Archivo':<45} Descripción")
    logging.info('  ' + '─' * 85)
    for f, d in outputs:
        logging.info(f"  {f:<45} {d}")
    logging.info(f"\n  Participantes             : {len(df)}")
    logging.info(f"  Texto libre excluido      : {len(VARS_TEXTO_LIBRE)}")
    logging.info(f"  Redundantes (modelado)    : {len(VARS_REDUNDANTES_ELIMINAR)}")
    logging.info(f"  Cat.2 VERDE (no eliminar) : "
                 f"{sum(1 for s in semaforos.values() if s=='VERDE')}")
    logging.info(f"\n  Siguiente paso: python pipeline_v9a_eda.py")


if __name__ == "__main__":
    main()